In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Lectura del archivo de candidatos
df = pd.read_csv('../data/candidates.csv')

# Criterio de contratación: puntuación de prueba y entrevista >= 7
df['is_hired'] = ((df['Code Challenge Score'] >= 7) & (df['Technical Interview'] >= 7)).astype(int)

# Limpieza básica de faltantes
df['Seniority'] = df['Seniority'].fillna('Desconocido')
df['Email'] = df['Email'].fillna('sin_correo@dominio.com')
df['Application Date'] = pd.to_datetime(df['Application Date'])

# Construcción de dimensiones
dim_tech = pd.DataFrame({'technology_name': df['Technology'].unique()}).reset_index(names='technology_id')
dim_seniority = pd.DataFrame({'seniority_level': df['Seniority'].unique()}).reset_index(names='seniority_id')
dim_location = pd.DataFrame({'country': df['Country'].unique()}).reset_index(names='location_id')

dim_date = pd.DataFrame({'full_date': df['Application Date'].drop_duplicates()}).sort_values('full_date').reset_index(drop=True)
dim_date['date_id'] = dim_date['full_date'].dt.strftime('%Y%m%d').astype(int)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['day'] = dim_date['full_date'].dt.day

dim_candidate = df[['First Name', 'Last Name', 'Email']].drop_duplicates().reset_index(drop=True)
dim_candidate.index.name = 'candidate_id'
dim_candidate = dim_candidate.reset_index()
dim_candidate['candidate_id'] = dim_candidate['candidate_id'] + 1

# Mapeo de IDs hacia la tabla de hechos
df['date_id'] = df['Application Date'].dt.strftime('%Y%m%d').astype(int)

df_merged = df.merge(dim_tech, left_on='Technology', right_on='technology_name') \
              .merge(dim_seniority, left_on='Seniority', right_on='seniority_level') \
              .merge(dim_location, left_on='Country', right_on='country') \
              .merge(dim_candidate, on=['First Name', 'Last Name', 'Email'])

fact_apps = df_merged[[
    'candidate_id', 'technology_id', 'seniority_id', 'location_id', 
    'date_id', 'Yoe', 'Code Challenge Score', 'Technical Interview', 'is_hired'
]].copy()
fact_apps.index.name = 'application_id'
fact_apps = fact_apps.reset_index()

# Persistencia en SQLite
db = create_engine('sqlite:///../dw_candidates.db')

dim_tech.to_sql('dim_technology', db, if_exists='replace', index=False)
dim_seniority.to_sql('dim_seniority', db, if_exists='replace', index=False)
dim_location.to_sql('dim_location', db, if_exists='replace', index=False)
dim_date.to_sql('dim_date', db, if_exists='replace', index=False)
dim_candidate.to_sql('dim_candidate', db, if_exists='replace', index=False)
fact_apps.to_sql('fact_applications', db, if_exists='replace', index=False)

print("Datos procesados y cargados correctamente en SQLite.")

Datos procesados y cargados correctamente en SQLite.


OperationalError: (sqlite3.OperationalError) no such column: t.technology
[SQL: 
   SELECT t.technology, COUNT(*) as contrataciones
   FROM fact_applications f
   JOIN dim_technology t ON f.technology_id = t.technology_id
   WHERE f.is_hired = 1
   GROUP BY t.technology ORDER BY contrataciones DESC
]
(Background on this error at: https://sqlalche.me/e/20/e3q8)